# FLAG 2027 — NB-53: SCALE-ID-01, more VoxCeleb2 data for v11's T3 head (end to end: extraction + gate + submission.zip)

**Why.** Identity count is our one proven lever (T3ENC-01, BIGB-01). FLAG_37 lost about 1,117 VoxCeleb2 dev speakers because
`vox2_mp4_1.zip` on the mirror is a TRUNCATED upload (27.0 GB vs ~50 GB). The same mirror holds the OFFICIAL split archives
(`vox2_dev_mp4_parta[a-i]`, `vox2_dev_aac_parta[a-h]`): intact zip64 files whose central directory sits in the last part
(checked 10/10). Every face track / voice clip is read by HTTP range, nothing is downloaded whole.
Registered in `Experiment/SCALE-ID-01/README.md` before any result.

**Extraction (section 0):** `new` = the missing dev speakers + the 120 VoxCeleb2 test speakers (6 faces / 8 voices each,
FLAG_37's rule); `more` = 6 more faces + 8 more voices for each of FLAG_37's 4,877 speakers (FLAG_43's design). Encoders:
FaRL16, CLIP-L, SimAM100 p / ft (T3 cell) + VoxLingua107 LID; a speaker with any clip of top-1 Bengali or p(bn) >= 0.2 loses
every clip (old rows included).

| arm | training rows | role |
|---|---|---|
| B | v11's head rows (v4 + MAV-Celeb + 4,852 VoxCeleb2) | same-device control |
| X / M | B + `new` / B + `more`, same epochs as B | diagnostics |
| **XM** | B + `new` + `more`, same epochs as B | **decision** (replaces v11's head) |

**Gate (default), computed here:** CA-FUSE(S6 + XM deep / cca + t3_relg) vs v11; sensitivity vs this run's B.
**Then:** final XM head (exported), dev streams, fuser refit, `submission.zip` (English = v11). **Submit only if PASSES.**

**Inputs (Add Input):** `flag2027-lab-data`, `flag2027-eda-bundle`, `flag2027-e2e-dev`. Optional: this notebook's previous
output (resume: extraction chunks, finished jobs). **Settings:** **GPU T4 x2**, **Internet ON** (Hugging Face, ModelScope,
GitHub). **Time:** extraction about 1.5-2.5 h (about 50k face tracks + 60k voice clips by range reads), then about 1 h of jobs.
**Output:** `/kaggle/working/feats_vox2b/` (keep it: a later run reuses it), `/kaggle/working/e2e_out/`, `/kaggle/working/submission.zip`.

In [ ]:
import os, sys, json, time, shutil, subprocess, hashlib, base64, io, zipfile, threading
from pathlib import Path
T_START = time.time()
LOCAL = os.environ.get("FLAG_LOCAL") == "1"
IN = Path(os.environ.get("FLAG_IN", "/kaggle/input"))
WORK = Path(os.environ.get("FLAG_OUT", "/kaggle/working"))
EXP = os.environ.get("LAB_EXP", "SCALE-ID-01")
TAG = "scaleid01"
EXP_CODE = ["Experiment/SCALE-ID-01/run.py", "Experiment/E2E/v11_head.py"]                                            # the experiment's own code: a resume needs the same files
FINAL_AFTER_REPORT = False                                      # True: the final head depends on the report (a chosen variant)
FOLDS = [int(x) for x in os.environ.get("LAB_FOLDS", "1,2,3,4,5").split(",") if x]
SPLIT = os.environ.get("E2E_SPLIT", "dev")                         # dev: flag2027-e2e-dev; eval: the components of a FLAG_50 run
BUDGET_H = float(os.environ.get("E2E_BUDGET_H", "10.5"))            # no new job starts after this many hours (Kaggle stops at 12)


def log(*a):
    print(time.strftime("%H:%M:%S"), f"[{(time.time() - T_START) / 3600:4.1f}h]", *a, flush=True)


mans = [(m.parent, json.loads(m.read_text())) for m in sorted(IN.rglob("lab_manifest.json"))]
main = [(d, m) for d, m in mans if m.get("kind", "data") == "data"]
assert len(main) == 1, f"attach exactly one flag2027-lab-data (found {len(main)})"
DATA, MAN = main[0]
ROOTS = [d for d, _ in mans]
eda = sorted(IN.rglob("eda_bundle_manifest.json"))
assert len(eda) == 1, f"attach exactly one flag2027-eda-bundle (found {len(eda)})"
created = json.loads(eda[0].read_text())["created"]
assert created == MAN["eda_bundle_created"], f"eda bundle {created} != the lab's {MAN['eda_bundle_created']}"
e2e = [(m.parent, json.loads(m.read_text())) for m in sorted(IN.rglob("e2e_manifest.json"))]
e2e = [(d, m) for d, m in e2e if m.get("split") == SPLIT and (d / f"components_{SPLIT}.npz").exists()]
assert len(e2e) >= 1, f"attach flag2027-e2e-dev (or, for E2E_SPLIT=eval, the output of FLAG_50 on the Evaluation data)"
E2E_DIR, E2E_MAN = e2e[0]
SP = WORK / "sp"; (SP / "out/safe04").mkdir(parents=True, exist_ok=True)
EXTRA_LINKS = {}                                                    # lab path -> folder made by this notebook (extraction cells)
OUTD = WORK / "e2e_out"; OUTD.mkdir(parents=True, exist_ok=True)
LAB = WORK / "lab"
import torch
NGPU = torch.cuda.device_count()
GPU = [torch.cuda.get_device_name(i) for i in range(NGPU)] or ["none (CPU: slow)"]
log(f"experiment {EXP} ({TAG}) | folds {FOLDS} | split {SPLIT} ({E2E_DIR}, built {E2E_MAN['created']}) | lab data built {MAN['created']} | "
    f"eda {created} | GPUs {GPU} | CPUs {os.cpu_count()} | torch {torch.__version__} | budget {BUDGET_H} h")

## 0. Extraction: more VoxCeleb2 data from the official split archives (resumable per chunk)

In [ ]:
import types
CACHE = Path(os.environ.get("FLAG_CACHE", "/tmp/flag_cache")); CACHE.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)

In [ ]:
# WeSpeaker (commit 9fecd6c, as Track 3) and openai CLIP (commit d50d76d, FaRL's code base): --no-deps, their pins would
# replace Kaggle's torch; the few runtime deps by hand.
if not LOCAL:
    !pip -q install ftfy regex kaldiio silero-vad av "speechbrain>=1.1.1" 2>&1 | tail -1
    !pip -q install --no-deps "git+https://github.com/wenet-e2e/wespeaker.git@9fecd6cb4f47475d01761d87c826298dff4ef18c" "git+https://github.com/openai/CLIP.git@d50d76daa670286dd6cacf3bcd80b5e4823fc8e1" 2>&1 | tail -2
else:                                           # local smoke test: wespeaker + its deps in an isolated folder
    for p_ in reversed([p for p in os.environ.get("FLAG_T3_PATH", "").split(";") if p]):
        sys.path.insert(0, p_)
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F, soundfile as sf, transformers
DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| transformers", transformers.__version__, "| device", DEV,
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")


def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)

In [ ]:
# ============================================================================ audio + batching
TARGET_SR = 16000
AUDIO_STATS = {"files": 0, "resampled": 0, "multichannel": 0, "short": 0}
MIN_SEC = 0.1                  # shorter clips are tiled to 0.1 s (w2v2 / AudioCLIP STFT fail on a few ms; v4 clips are >= 2 s)
BROKEN = {"audio": [], "image": []}   # unreadable sources: scored from a stand-in, listed in the manifest


def _src_name(src):
    """Full path of a file source (base names repeat across splits); the archive member name of a zip stream."""
    if isinstance(src, (bytes, bytearray)):
        return f"<{len(src)} bytes>"
    return str(src) if isinstance(src, (str, Path)) else str(getattr(src, "name", None) or src)


def load_audio(src):
    """Mono float32 at 16 kHz as the organiser loads audio (librosa.load(sr=16000, mono=True): channel mean + soxr_hq).
    All v4 wavs are 16 kHz mono; MAV-Celeb v1 has 44.1 kHz files. `src` = path or file-like.
    A file that cannot be read (or is empty / non-finite) -> 1 s of silence, listed in BROKEN["audio"]."""
    try:
        w, sr = sf.read(src if hasattr(src, "read") else str(src), dtype="float32", always_2d=True)
        if w.size == 0 or not np.isfinite(w).all():
            raise ValueError(f"{w.shape[0]} samples, finite {bool(np.isfinite(w).all())}")
    except Exception as e:
        BROKEN["audio"].append(f"{_src_name(src)} | {type(e).__name__}: {e}")
        log("!! unreadable audio -> 1 s of silence:", BROKEN["audio"][-1])
        return np.zeros(TARGET_SR, np.float32)
    AUDIO_STATS["files"] += 1
    if w.shape[1] > 1:
        AUDIO_STATS["multichannel"] += 1
        w = w.mean(1, keepdims=True)
    w = w[:, 0]
    if sr != TARGET_SR:
        import librosa
        w = librosa.resample(w, orig_sr=sr, target_sr=TARGET_SR).astype(np.float32)
        AUDIO_STATS["resampled"] += 1
    if len(w) < int(MIN_SEC * TARGET_SR):
        AUDIO_STATS["short"] += 1
        w = np.tile(w, int(np.ceil(MIN_SEC * TARGET_SR / len(w))))[:int(MIN_SEC * TARGET_SR)]
    return w


def centre_crop(w, L):
    if len(w) <= L:
        return w
    s = (len(w) - L) // 2
    return w[s:s + L]


def pack_batches(durs, max_padded_sec=240.0, max_bs=32):
    """Longest first; a batch costs len(batch) * longest item after padding."""
    order = np.argsort(-np.asarray(durs), kind="stable")
    batches, cur = [], []
    for i in order:
        longest = durs[cur[0]] if cur else durs[i]
        if cur and ((len(cur) + 1) * longest > max_padded_sec or len(cur) >= max_bs):
            batches.append(cur); cur = []
        cur.append(int(i))
    if cur:
        batches.append(cur)
    return batches


def run_batched(items, fn, durs, max_padded_sec=240.0, max_bs=32, cpu_fn=None, label=""):
    """fn(list of items) -> list of outputs, in item order. A CUDA OOM halves the batch; a single item goes to CPU."""
    out = [None] * len(items)
    t0, done_n = time.time(), 0

    def go(idx):
        try:
            res = fn([items[i] for i in idx])
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache()
            if len(idx) == 1:
                res = (cpu_fn or fn)([items[idx[0]]])
            else:
                h = len(idx) // 2
                go(idx[:h]); go(idx[h:])
                return
        for i, r in zip(idx, res):
            out[i] = r

    batches = pack_batches(durs, max_padded_sec, max_bs)
    for bi, b in enumerate(batches):
        go(b)
        done_n += len(b)
        if bi % 50 == 0 or done_n == len(items):
            el = time.time() - t0
            log(f"  {label} {done_n}/{len(items)}  {el:.0f}s  eta {el / done_n * (len(items) - done_n):.0f}s")
    return out


def cos_rows(a, b):
    a, b = np.asarray(a, np.float64), np.asarray(b, np.float64)
    return np.sum(a * b, 1) / (np.linalg.norm(a, axis=1) * np.linalg.norm(b, axis=1) + 1e-12)


def amp(flag):
    return torch.autocast("cuda", dtype=torch.float16, enabled=bool(flag) and DEV.type == "cuda")


def read_rgb(src):
    """PIL RGB image; an image that cannot be read -> a black 224 x 224 image, listed in BROKEN["image"]."""
    from PIL import Image
    try:
        return Image.open(io.BytesIO(src) if isinstance(src, (bytes, bytearray)) else src).convert("RGB")
    except Exception as e:
        BROKEN["image"].append(f"{_src_name(src)} | {type(e).__name__}: {e}")
        log("!! unreadable image -> black image:", BROKEN["image"][-1])
        return Image.new("RGB", (224, 224))

In [ ]:
# ============================================================================ T3ENC-01 encoders: the encoder family of Track 3's VoxBlink system, frozen
# Track 3 (a teammate's VoxBlink pipeline, CB Bangla ng ~16 / g ~25) used these four encoders; here they run on OUR rows
# (MAV-Celeb v4 train, v1/v2, dev lists; and VoxCeleb2 in FLAG_41), so we can test locally whether the encoders and a
# cross-modal head carry the gain WITHOUT VoxBlink data (docs/PLAN_VOXBLINK_2026-10-07.md, V1). Every sample on its own.
#   farl16V   FaRL ViT-B/16 LAION-Face 20M **ep16** (github.com/FacePerceiver/FaRL release, 652 MB) on openai CLIP ViT-B/16
#             (CLIP preprocessing: 224 bicubic, centre crop); 512-d image embedding (our existing `farl` is the ep64 copy)
#   clipLV    openai/clip-vit-large-patch14 @ 32bd642 (transformers): get_image_features, 768-d
#   sam100pA  WeSpeaker voxblink2_samresnet100 (VoxBlink2 pretrained), official ModelScope zip (wenet/wespeaker_pretrained_models)
#   sam100fA  WeSpeaker voxblink2_samresnet100_ft (VoxBlink2 pretrained -> VoxCeleb2 large-margin fine-tune)
#             both: wespeaker fbank + CMN, int16-scaled waveform (as our wssim34 / wsr221 cells), 256-d
import zipfile as _zf

FARL16_URL = "https://github.com/FacePerceiver/FaRL/releases/download/pretrained_weights/FaRL-Base-Patch16-LAIONFace20M-ep16.pth"
FARL16_SIZE = 652165542
CLIPL_ID, CLIPL_REV = "openai/clip-vit-large-patch14", "32bd64288804d66eefd0ccbe215aa642df71cc41"
WESPK_TREE = "https://modelscope.cn/api/v1/datasets/wenet/wespeaker_pretrained_models/oss/tree"
SAM_ZIPS = {"sam100pA": ("voxblink2_samresnet100.zip", 186890339), "sam100fA": ("voxblink2_samresnet100_ft.zip", 186890839)}
T3_STATS = {}


def fetch(url, dest, size=None):
    dest = Path(dest)
    if dest.exists() and (size is None or dest.stat().st_size == size):
        return dest
    torch.hub.download_url_to_file(url, str(dest) + ".part", progress=False)
    Path(str(dest) + ".part").replace(dest)
    assert size is None or dest.stat().st_size == size, f"{dest.name}: {dest.stat().st_size} bytes, expected {size}"
    return dest


class FaRL16Face:
    kind = "face"

    def __init__(self):
        import clip
        m, self.pre = clip.load("ViT-B/16", device="cpu", download_root=str(CACHE / "clip"))
        sd = torch.load(fetch(FARL16_URL, CACHE / "FaRL-ep16.pth", FARL16_SIZE), map_location="cpu", weights_only=False)["state_dict"]
        vis = [k for k in m.state_dict() if k.startswith("visual.")]
        hit = [k for k in vis if k in sd and sd[k].shape == m.state_dict()[k].shape]
        assert len(hit) == len(vis), f"FaRL ep16 covers {len(hit)}/{len(vis)} visual tensors"
        m.load_state_dict(sd, strict=False)
        self.m = m.visual.float().to(DEV).eval()

    def prep(self, im):
        return self.pre(im)

    @torch.no_grad()
    def __call__(self, x, use_amp=True):
        return {"": self.m(x.to(DEV)).float().cpu().numpy()}


class CLIPLFace:
    kind = "face"

    def __init__(self):
        from transformers import CLIPModel, CLIPImageProcessor
        self.proc = CLIPImageProcessor.from_pretrained(CLIPL_ID, revision=CLIPL_REV)
        self.m = CLIPModel.from_pretrained(CLIPL_ID, revision=CLIPL_REV, torch_dtype=torch.float32).to(DEV).eval()

    def prep(self, im):
        return self.proc(images=im.convert("RGB"), return_tensors="pt").pixel_values[0]

    @torch.no_grad()
    def __call__(self, x, use_amp=True):
        f = self.m.get_image_features(pixel_values=x.to(DEV))
        f = f.pooler_output if hasattr(f, "pooler_output") else f          # transformers 5 may return an output object
        return {"": f.float().cpu().numpy()}


def wespeaker_dir(zip_name, size):
    """The official zip (ModelScope, as wespeaker's own hub does), unpacked without the __MACOSX entries."""
    d = CACHE / "wespeaker" / zip_name[:-4]
    if (d / "avg_model.pt").exists() and (d / "config.yaml").exists():
        return d
    d.mkdir(parents=True, exist_ok=True)
    import requests
    tree = requests.get(WESPK_TREE, timeout=120).json()["Data"]
    url = [it["Url"] for it in tree if it.get("Key") == zip_name][0]
    z = fetch(url, d / zip_name, size)
    with _zf.ZipFile(z) as zf:
        for mem in zf.namelist():
            if not mem.startswith("__MACOSX") and not mem.endswith("/"):
                (d / Path(mem).name).write_bytes(zf.read(mem))
    # some releases (voxceleb_resnet293_LM) name the pair <model>.pt / <model>.yaml instead of avg_model.pt / config.yaml
    pts, ymls = sorted(d.glob("*.pt")), sorted(d.glob("*.yaml"))
    if not (d / "avg_model.pt").exists() and len(pts) == 1:
        pts[0].rename(d / "avg_model.pt")
    if not (d / "config.yaml").exists() and len(ymls) == 1:
        ymls[0].rename(d / "config.yaml")
    assert (d / "avg_model.pt").exists() and (d / "config.yaml").exists(), f"{zip_name}: unexpected content {zf.namelist()[:6]}"
    return d


class SimAMVoice:
    kind = "voice"

    def __init__(self, name):
        for mod, attr in (("wespeaker.frontend.s3prl", "S3prlFrontend"), ("wespeaker.frontend.whisper_encoder", "whisper_encoder"),
                          ("wespeaker.frontend.w2vbert", "W2VBertFrontend"), ("wespeaker.diar.umap_clusterer", "cluster"),
                          ("wespeaker.diar.extract_emb", "subsegment")):
            if mod not in sys.modules:                     # SSL frontends / diarisation helpers: not used by fbank models
                stub = types.ModuleType(mod); setattr(stub, attr, None); sys.modules[mod] = stub
        import wespeaker
        self.m = wespeaker.load_model(str(wespeaker_dir(*SAM_ZIPS[name])))
        self.m.set_device("cuda:0" if DEV.type == "cuda" else "cpu")
        T3_STATS[name] = {"params_M": round(sum(p.numel() for p in self.m.model.parameters()) / 1e6, 1)}

    @torch.no_grad()
    def __call__(self, w, use_amp=True):
        pcm = torch.from_numpy(np.ascontiguousarray(w, dtype=np.float32) * 32768.0)[None]
        feats = self.m.compute_features(pcm, sample_rate=16000, cmn=True)
        out = self.m.model(feats.to(self.m.device))
        out = out[-1] if isinstance(out, tuple) else out
        return {"": out[0].reshape(-1).float().cpu().numpy()}


NEW = {"farl16V": FaRL16Face, "clipLV": CLIPLFace, "sam100pA": lambda: SimAMVoice("sam100pA"),
       "sam100fA": lambda: SimAMVoice("sam100fA")}


def build(name):
    return NEW[name]()


def release(keep_ib=False):
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


ARRAYS_NEW = {"farl16V": ("face", {"": "farl16"}), "clipLV": ("face", {"": "clipL"}),
              "sam100pA": ("voice", {"": "sam100p"}), "sam100fA": ("voice", {"": "sam100f"})}
MAIN_NEW = {k: "" for k in NEW}
RESHAPE = {}

In [ ]:
if SPLIT != "dev":
    log("split {}: VoxCeleb2 extraction skipped (the dev run's final head is reloaded)".format(SPLIT))
else:
    # ============================================================================ SCALE-ID-01: the VoxCeleb2 speakers our T3 head never saw
    # FLAG_37 read faces from ProgramComputer/voxceleb vox2_mp4_{2..6}.zip: vox2_mp4_1.zip is a TRUNCATED upload (27.0 GB where the
    # others hold ~50 GB; 244 GB in all vs 267 GB for the official archive), so its central directory is missing (BadZipFile) and
    # ~1,117 dev speakers were lost. The same mirror holds the OFFICIAL split archives (vox2_dev_mp4_parta[a-i], 267 GB, and
    # vox2_dev_aac_parta[a-h], 77 GB): byte chunks of one zip64 file whose central directory sits in the last part (checked
    # 10/10: 1,243,744 / 1,243,573 entries). Every member is read by HTTP range on the right part(s); nothing is downloaded whole.
    # Two additions (column `kind`):
    #   new   VoxCeleb2 dev speakers absent from FLAG_37's key list + the 120 VoxCeleb2 test speakers, sampled with FLAG_37's rule:
    #         NF = 6 face tracks (first track of 6 videos spread over the speaker's videos, middle frame of the 224 x 224 track),
    #         NV = 8 voice clips (<= 2 per video, 12 s centre crop)
    #   more  FLAG_37's speakers: 6 MORE_IDS face tracks (first tracks of videos FLAG_37 did not use, then second tracks of used ones)
    #         and 8 MORE_IDS voice clips (not FLAG_37's; <= 2 per video counting FLAG_37's clips) = FLAG_43's design
    # Encoders = the T3 cell (FaRL16,
    # CLIP-L, SimAM100 p / ft) + VoxLingua107 LID (FLAG_42) with the registered Bengali rule (drop every clip of a speaker with any
    # clip of top-1 Bengali or p(bn) >= 0.2). Resumable per chunk (attach the previous output).
    import requests, struct, zlib, threading
    import concurrent.futures as cf

    HF = "https://huggingface.co/datasets/ProgramComputer/voxceleb/resolve/main/vox2/"
    SOURCES = {"dev_mp4": [f"vox2_dev_mp4_parta{c}" for c in "abcdefghi"], "dev_aac": [f"vox2_dev_aac_parta{c}" for c in "abcdefgh"],
               "test_mp4": ["vox2_test_mp4.zip"], "test_aac": ["vox2_test_aac.zip"]}
    NF, NV, PER_VIDEO, CLIP_SEC, THREADS = 6, 8, 2, 12.0, 16
    VB = WORK / "feats_vox2b"
    FP_, VP_ = VB / "parts_face", VB / "parts_voice"
    for d in (VB, FP_, VP_):
        d.mkdir(parents=True, exist_ok=True)
    SMOKE_IDS = int(os.environ.get("VOX_SMOKE_IDS", "0")) if (LOCAL or os.environ.get("FLAG_SMOKE") == "1") else 0   # FLAG_SMOKE: Kaggle CPU smoke (kapi --smoke)
    for prev in ([] if LOCAL else IN.rglob("feats_vox2b")):          # resume: chunks and listings of an earlier run
        if VB in prev.parents or prev == VB:
            continue
        for f in list(prev.glob("parts_*/*.npz")) + list(prev.glob("listing_*.parquet")):
            tgt = VB / f.relative_to(prev)
            if not tgt.exists():
                tgt.parent.mkdir(parents=True, exist_ok=True); shutil.copy(f, tgt)


    class Remote:
        """A remote zip made of byte-chunk parts (or one part), read by absolute offset with HTTP range requests."""
        def __init__(self, names):
            self.names, self.urls, self.sizes = names, {}, []
            for n in names:
                with requests.get(HF + n, headers={"Range": "bytes=0-0"}, allow_redirects=True, timeout=120, stream=True) as r:
                    r.raise_for_status()
                    self.urls[n] = r.url
                    self.sizes.append(int(r.headers["Content-Range"].split("/")[-1]))
            self.starts = np.r_[0, np.cumsum(self.sizes)]
            self.tl = threading.local()

        def _get(self, j, a, b):
            n = self.names[j]
            if not hasattr(self.tl, "s"):
                self.tl.s = requests.Session()
            for attempt in range(5):
                try:
                    r = self.tl.s.get(self.urls[n], headers={"Range": f"bytes={a}-{b}"}, timeout=300)
                    if r.status_code in (401, 403):
                        with requests.get(HF + n, headers={"Range": "bytes=0-0"}, allow_redirects=True, timeout=120, stream=True) as q:
                            self.urls[n] = q.url
                        continue
                    r.raise_for_status()
                    assert len(r.content) == b - a + 1, (len(r.content), b - a + 1)
                    return r.content
                except Exception:
                    time.sleep(3 * (attempt + 1))
            raise RuntimeError(f"range {a}-{b} of {n} failed")

        def read(self, off, n):
            """n bytes at absolute offset off (may span parts)."""
            out, end = [], off + n
            while off < end:
                j = int(np.searchsorted(self.starts, off, side="right") - 1)
                a = off - self.starts[j]; b = min(end, self.starts[j + 1]) - self.starts[j] - 1
                out.append(self._get(j, int(a), int(b))); off = int(self.starts[j] + b + 1)
            return b"".join(out)

        def listing(self):
            """Central directory -> DataFrame(name, off, csize, method) of every file member."""
            total = int(self.starts[-1])
            tail = self.read(total - 65536, 65536)
            i = tail.rfind(b"PK\x06\x06")
            if i >= 0:
                cs, co = struct.unpack("<QQ", tail[i + 40:i + 56])
            else:
                i = tail.rfind(b"PK\x05\x06"); cs, co = struct.unpack("<II", tail[i + 12:i + 20])
            cd = self.read(co, cs)
            rows, p = [], 0
            while p + 46 <= len(cd) and cd[p:p + 4] == b"PK\x01\x02":
                method, = struct.unpack_from("<H", cd, p + 10)
                csz, usz = struct.unpack_from("<II", cd, p + 20)
                n, m, k = struct.unpack_from("<HHH", cd, p + 28)
                off, = struct.unpack_from("<I", cd, p + 42)
                name = cd[p + 46:p + 46 + n].decode("utf-8", "replace")
                ex = cd[p + 46 + n:p + 46 + n + m]
                if 0xFFFFFFFF in (csz, usz, off):                          # zip64 extra field: usize, csize, offset (only those set)
                    q = 0
                    while q + 4 <= len(ex):
                        hid, hsz = struct.unpack_from("<HH", ex, q)
                        if hid == 1:
                            vals, r_ = [], q + 4
                            for v in (usz, csz, off):
                                if v == 0xFFFFFFFF:
                                    vals.append(struct.unpack_from("<Q", ex, r_)[0]); r_ += 8
                                else:
                                    vals.append(v)
                            usz, csz, off = vals
                            break
                        q += 4 + hsz
                if not name.endswith("/"):
                    rows.append((name, off, csz, method))
                p += 46 + n + m + k
            return pd.DataFrame(rows, columns=["name", "off", "csize", "method"])

        def member(self, off, csize, method):
            h = self.read(off, 30)
            assert h[:4] == b"PK\x03\x04", "bad local header"
            n, e = struct.unpack("<HH", h[26:30])
            d = self.read(off + 30 + n + e, csize)
            return zlib.decompress(d, -15) if method == 8 else d


    REM, LST = {}, {}
    for key, names in SOURCES.items():
        REM[key] = Remote(names)
        f = VB / f"listing_{key}.parquet"
        if f.exists():
            L_ = pd.read_parquet(f)
        else:
            t0 = time.time(); L_ = REM[key].listing()
            ext = ".mp4" if key.endswith("mp4") else ".m4a"
            L_ = L_[L_.name.str.endswith(ext)].reset_index(drop=True)
            sp = L_.name.str.split("/")
            L_["spk"], L_["video"], L_["utt"] = sp.str[-3], sp.str[-2], sp.str[-1].str[:-4]
            L_.to_parquet(f)
            log(f"{key}: central directory {len(L_)} members [{time.time() - t0:.0f}s]")
        L_["src"] = key
        LST[key] = L_

    # ---------------------------------------------------------------- identities and samples
    meta = pd.read_csv(io.BytesIO(requests.get(HF + "vox2_meta.csv", timeout=120).content), dtype=str)
    meta.columns = [c.strip().lstrip("﻿") for c in meta.columns]
    meta = meta.apply(lambda c: c.str.strip())
    kroot = next(r / "Experiment/VOX2-01/feats_vox2" for r in ROOTS if (r / "Experiment/VOX2-01/feats_vox2/face_meta.csv").exists())
    OF = pd.read_csv(kroot / "face_meta.csv", dtype=str); OV = pd.read_csv(kroot / "voice_meta.csv", dtype={"spk": str, "video": str, "utt": str})
    OLD = sorted(set(OF.spk))
    FL = pd.concat([LST["dev_mp4"], LST["test_mp4"]], ignore_index=True).sort_values(["spk", "video", "utt"])
    AL = pd.concat([LST["dev_aac"], LST["test_aac"]], ignore_index=True).sort_values(["spk", "video", "utt"])
    NEW_IDS = sorted((set(meta[meta.Set == "dev"]["VoxCeleb2 ID"]) - set(OLD)) | set(meta[meta.Set == "test"]["VoxCeleb2 ID"]))
    NEW_IDS = sorted(set(NEW_IDS) & set(FL.spk) & set(AL.spk))
    MORE_IDS = sorted(set(OLD) & set(FL.spk) & set(AL.spk))
    if SMOKE_IDS:
        NEW_IDS, MORE_IDS = NEW_IDS[:SMOKE_IDS], MORE_IDS[:SMOKE_IDS]
    log(f"identities: new {len(NEW_IDS)} (VoxCeleb2 dev outside FLAG_37's {len(OLD)} + test), more samples for {len(MORE_IDS)}")


    def pick(n, k):
        return list(range(n)) if n <= k else [int(round(x)) for x in np.linspace(0, n - 1, k)]


    used_fv = set(zip(OF.spk, OF.video)); used_f = set(zip(OF.spk, OF.video, OF.utt)); used_v = set(zip(OV.spk, OV.video, OV.utt))
    vcount = OV.groupby(["spk", "video"]).size().to_dict()
    FSEL, VSEL = [], []
    fl_new, fl_more = FL[FL.spk.isin(set(NEW_IDS))], FL[FL.spk.isin(set(MORE_IDS))]
    for spk, g in fl_new.groupby("spk", sort=True):                   # new: FLAG_37's rule
        first = g.drop_duplicates("video")
        FSEL += [dict(first.iloc[i], kind="new") for i in pick(len(first), NF)]
    for spk, g in fl_more.groupby("spk", sort=True):                  # more: unused videos first, then second tracks of used ones
        g = g.reset_index(drop=True)
        firstmask = ~g.duplicated("video").values
        first = g[firstmask]
        unused = first[[(spk, v) not in used_fv for v in first.video]]
        cand = [dict(unused.iloc[i], kind="more") for i in pick(len(unused), NF)]
        if len(cand) < NF:
            rest = g[~firstmask]
            rest = rest[[(spk, v, u) not in used_f for v, u in zip(rest.video, rest.utt)]].drop_duplicates("video")
            cand += [dict(r, kind="more") for _, r in rest.head(NF - len(cand)).iterrows()]
        FSEL += cand
    FSEL = pd.DataFrame(FSEL).reset_index(drop=True)
    for kind, ids in (("new", NEW_IDS), ("more", MORE_IDS)):
        for spk, g in AL[AL.spk.isin(set(ids))].groupby("spk", sort=True):
            byv = []
            for vid, v in g.groupby("video", sort=True):
                rows = [r for r in v.itertuples(index=False) if (spk, vid, r.utt) not in used_v]
                room = PER_VIDEO - vcount.get((spk, vid), 0)        # <= 2 clips per video, FLAG_37's clips included
                byv.append(rows[:max(0, room)])
            cand = [r for k in range(PER_VIDEO) for v in byv if k < len(v) for r in [v[k]]]
            VSEL += [dict(r._asdict(), kind=kind) for r in cand[:NV + 4]]  # spares for unreadable / empty clips
    VSEL = pd.DataFrame(VSEL).reset_index(drop=True)
    log(f"faces: {len(FSEL)} tracks ({FSEL.kind.value_counts().to_dict()}); voice candidates: {len(VSEL)} ({VSEL.kind.value_counts().to_dict()})")


    def middle_frame(mp4):
        import av
        with av.open(io.BytesIO(mp4)) as c:
            st = c.streams.video[0]; n = st.frames or 50
            last = None
            for k, fr in enumerate(c.decode(st)):
                last = fr
                if k >= n // 2:
                    break
            return last.to_image().convert("RGB") if last is not None else None


    def decode_m4a(b):
        import av
        out = []
        with av.open(io.BytesIO(b)) as c:
            res = av.AudioResampler(format="flt", layout="mono", rate=TARGET_SR)
            for fr in c.decode(c.streams.audio[0]):
                out += [o.to_ndarray().reshape(-1) for o in res.resample(fr)]
            out += [o.to_ndarray().reshape(-1) for o in res.resample(None)]
        return np.concatenate(out).astype(np.float32) if out else np.zeros(TARGET_SR // 10, np.float32)


    def face_job(r):
        try:
            return middle_frame(REM[r.src].member(int(r.off), int(r.csize), int(r.method)))
        except Exception as e:
            BROKEN["image"].append(f"{r.name} | {type(e).__name__}: {e}"); return None


    def voice_job(r):
        try:
            return decode_m4a(REM[r.src].member(int(r.off), int(r.csize), int(r.method)))
        except Exception as e:
            BROKEN["audio"].append(f"{r.name} | {type(e).__name__}: {e}"); return None


    # ---------------------------------------------------------------- faces (chunks of 1 200 tracks)
    FENC = None
    CH = 1200
    for c0 in range(0, len(FSEL), CH):
        part = FP_ / f"faces_{c0 // CH:03d}.npz"
        if part.exists():
            continue
        t0 = time.time(); S_ = FSEL.iloc[c0:c0 + CH]
        with cf.ThreadPoolExecutor(THREADS) as ex:
            imgs = list(ex.map(face_job, S_.itertuples(index=False)))
        ok = [i for i, im in enumerate(imgs) if im is not None]
        if FENC is None:
            FENC = {"farl16": build("farl16V"), "clipL": build("clipLV")}
        R_ = {n: [] for n in FENC}
        for s in range(0, len(ok), 64):
            b = [imgs[i] for i in ok[s:s + 64]]
            for n, enc in FENC.items():
                R_[n].append(enc(torch.stack([enc.prep(im) for im in b]))[""])
        np.savez(part, spk=S_.spk.to_numpy(dtype=str)[ok], video=S_.video.to_numpy(dtype=str)[ok], utt=S_.utt.to_numpy(dtype=str)[ok],
                 kind=S_.kind.to_numpy(dtype=str)[ok], **{n: np.concatenate(v).astype(np.float16) for n, v in R_.items() if v})
        log(f"  faces chunk {c0 // CH}: {len(ok)}/{len(S_)} [{time.time() - t0:.0f}s]")
    FENC = None
    release()

    # ---------------------------------------------------------------- voices (chunks of 120 speakers): SimAM p / ft + LID
    from speechbrain.inference.classifiers import EncoderClassifier
    try:
        from speechbrain.utils.fetching import LocalStrategy
        _ls = {"local_strategy": LocalStrategy.COPY}
    except ImportError:
        _ls = {}
    VENC, LIDM = None, None
    spks = sorted(VSEL.spk.unique()); CHS = 120
    for c0 in range(0, len(spks), CHS):
        part = VP_ / f"voices_{c0 // CHS:03d}.npz"
        if part.exists():
            continue
        t0 = time.time()
        S_ = VSEL[VSEL.spk.isin(set(spks[c0:c0 + CHS]))].reset_index(drop=True)
        with cf.ThreadPoolExecutor(THREADS) as ex:
            waves = list(ex.map(voice_job, S_.itertuples(index=False)))
        keep, cnt = [], {}
        for i, w in enumerate(waves):
            s = S_.spk[i]
            if w is None or len(w) < int(MIN_SEC * TARGET_SR) or cnt.get(s, 0) >= NV:
                continue
            keep.append(i); cnt[s] = cnt.get(s, 0) + 1
        if VENC is None:
            VENC = {"sam100p": build("sam100pA"), "sam100f": build("sam100fA")}
            LIDM = EncoderClassifier.from_hparams(source="speechbrain/lang-id-voxlingua107-ecapa", savedir=str(CACHE / "lid_voxlingua107"),
                                                  run_opts={"device": str(DEV)}, **_ls)
            LIDM.eval()
        L = int(CLIP_SEC * TARGET_SR)
        crops = [centre_crop(waves[i], L) for i in keep]
        R_ = {n: np.stack([enc(w)[""] for w in crops]) for n, enc in VENC.items()}
        with torch.no_grad():
            R_["lid"] = np.stack([torch.softmax(LIDM.classify_batch(torch.from_numpy(np.ascontiguousarray(w))[None].to(DEV))[0].float().reshape(1, -1), 1)[0].cpu().numpy()
                                  for w in crops])
        np.savez(part, spk=S_.spk.to_numpy(dtype=str)[keep], video=S_.video.to_numpy(dtype=str)[keep], utt=S_.utt.to_numpy(dtype=str)[keep],
                 kind=S_.kind.to_numpy(dtype=str)[keep], dur=np.array([len(waves[i]) / TARGET_SR for i in keep], np.float32), **{n: v.astype(np.float16) for n, v in R_.items()})
        log(f"  voices chunk {c0 // CHS}: {len(keep)} clips of {len(set(S_.spk.values[keep]))} speakers [{time.time() - t0:.0f}s]")
    VENC = None
    release()

    # ---------------------------------------------------------------- assemble, LID rule, manifest
    def cat(parts, keys):
        Z = [np.load(p, allow_pickle=True) for p in sorted(parts)]            # this notebook's own part files
        Z = [z for z in Z if len(z["spk"])]
        return {k: (lambda a: a.astype(str) if a.dtype == object else a)(np.concatenate([z[k] for z in Z])) for k in keys}


    Fz = cat(FP_.glob("faces_*.npz"), ["spk", "video", "utt", "kind", "farl16", "clipL"])
    Vz = cat(VP_.glob("voices_*.npz"), ["spk", "video", "utt", "kind", "dur", "sam100p", "sam100f", "lid"])
    pd.DataFrame({k: Fz[k] for k in ("spk", "video", "utt", "kind")}).to_csv(VB / "face_meta.csv", index=False)
    pd.DataFrame({k: Vz[k] for k in ("spk", "video", "utt", "kind", "dur")}).to_csv(VB / "voice_meta.csv", index=False)
    for k in ("farl16", "clipL"):
        np.save(VB / f"face_{k}.npy", Fz[k])
    for k in ("sam100p", "sam100f"):
        np.save(VB / f"voice_{k}.npy", Vz[k])
    np.save(VB / "voice_lid_probs.npy", Vz["lid"])
    labels = [LIDM.hparams.label_encoder.ind2lab[i] for i in range(len(LIDM.hparams.label_encoder.ind2lab))] if LIDM is not None else None
    if labels is None:                                              # every voice chunk came from an earlier run: labels from the hub
        LIDM = EncoderClassifier.from_hparams(source="speechbrain/lang-id-voxlingua107-ecapa", savedir=str(CACHE / "lid_voxlingua107"),
                                              run_opts={"device": "cpu"}, **_ls)
        labels = [LIDM.hparams.label_encoder.ind2lab[i] for i in range(len(LIDM.hparams.label_encoder.ind2lab))]
    BN = [i for i, l in enumerate(labels) if l.split(":")[0].strip() == "bn"]
    assert len(labels) == 107 and len(BN) == 1
    P_ = Vz["lid"].astype(np.float32); top = P_.argmax(1); pbn = P_[:, BN[0]]
    flag = (top == BN[0]) | (pbn >= 0.2)
    drop = sorted(set(Vz["spk"][flag]))
    lid = dict(clips=int(len(P_)), clips_flagged=int(flag.sum()), speakers=int(len(set(Vz["spk"]))), speakers_dropped=len(drop),
               clips_dropped=int(np.isin(Vz["spk"], drop).sum()), speakers_dropped_ids=drop,
               top_languages=pd.Series([labels[i].split(":")[0] for i in top]).value_counts().head(15).to_dict())
    (VB / "lid_summary.json").write_text(json.dumps(lid, indent=1))
    VMAN = dict(created=time.strftime("%Y-%m-%d %H:%M"), new_identities=len(NEW_IDS), more_identities=len(MORE_IDS), faces=int(len(Fz["spk"])), voices=int(len(Vz["spk"])),
                face_speakers=int(len(set(Fz["spk"]))), voice_speakers=int(len(set(Vz["spk"]))), lid={k: v for k, v in lid.items() if k != "speakers_dropped_ids"},
                sources=SOURCES, sizes={k: [int(x) for x in REM[k].sizes] for k in REM}, broken={k: len(v) for k, v in BROKEN.items()},
                by_kind={"faces": pd.Series(Fz["kind"]).value_counts().to_dict(), "voices": pd.Series(Vz["kind"]).value_counts().to_dict()},
                rule="new = VoxCeleb2 dev outside FLAG_37's list + test (NF 6 / NV 8, <= 2 per video); more = 6 faces + 8 voices more for "
                     "FLAG_37's speakers (unused videos first; <= 2 clips per video counting FLAG_37's); LID rule of FLAG_42 over these clips")
    (VB / "feats_manifest.json").write_text(json.dumps(VMAN, indent=1))
    print(json.dumps(VMAN, indent=1))
    EXTRA_LINKS["Experiment/SCALE-ID-01/feats_vox2b"] = VB

## 1. Code (embedded) + data links + SAFE-04 reference tables

In [ ]:
# embedded code (= the local files, md5 below) -> /kaggle/working/lab; data folders linked next to it; SAFE-04 reference tables
CODE_B64 = "..."
CODE_MD5 = {
 "kaggle/build_l0_notebooks.py": "7bb1d2e1fc45651302f8d2f79a959a3e",
 "Experiment/SAFE-04/lab_streams.py": "e288a49c7f77f9a22e9cce00aead459b",
 "Experiment/SAFE-04/rel_sweep.py": "7166b20f70f25f342a22bfcadc5883ac",
 "Experiment/SAFE-04/rel_sweep2.py": "dd28762522bbaebbe1a42fc541c71464",
 "Experiment/SAFE-04/rel_sweep5.py": "557afcb255ab4f1b4696296633d87a88",
 "Experiment/SAFE-04/rel.py": "402aaf561639e75409f2ead79d78a939",
 "Experiment/VOX2-01/run_bridge.py": "e3260d3276cfdca8eb08dbcfd26f19aa",
 "Experiment/T3ENC-01/run.py": "78846fc22a0f1277e66b8bce266e1067",
 "Experiment/ALLNEG-01/run.py": "413e0779662fc2be0b5fb75930129e5b",
 "Experiment/BIGB-01/run.py": "f2a9a4f099496905ea93d9cf44025860",
 "Experiment/SCALE-VB/run.py": "2e2913fb5310c53191ead982de51582e",
 "Experiment/VBSEL-01/run.py": "2e99d3b90dc56675ca4b36ded1a83e96",
 "Experiment/VBSEL-01/vox2_sa_speakers.json": "37f4e8670949bc32f086003a593f9036",
 "Experiment/ENBASE-01/run.py": "4a40fb17c9585c2f66b177f5b529b777",
 "Experiment/SAFE-04/stream_gate.py": "079c789b6461024c007a0ffbfbcf7b7d",
 "Experiment/SAFE-04/lab_fuse.py": "fa8ce8f652bf3ee13597e872fda40b83",
 "Experiment/STACK-02/run.py": "3709cf4c175e60945144ec7c4198dbd0",
 "Experiment/STACK-02/pools.py": "cd867144bf43f0283003a8f1ba685cc0",
 "Experiment/STACK-02/eligible.json": "bbab2a531702ab631652d21e25a825d2",
 "Experiment/STACK-02/fold_persons.json": "aede157945600732e0f8a8337b8df0b5",
 "Experiment/E2E/e2e_lib.py": "0080f8eeaafea21af6a28f10f9f2d22d",
 "Experiment/SCALE-ID-01/run.py": "643068fd309dc3217c90a4e31efa08a4",
 "Experiment/E2E/v11_head.py": "005c368ec3f91ea229f406a8b47575ec",
 "lab_entry.py": "969ad42a47449a3a239cd9323a455f5a"
}
DATA_DIRS = ["kaggle/output/feats_av", "kaggle/output/feats_eval", "kaggle/output/feats_models", "Experiment/T3ENC-01/feats_t3enc", "Experiment/T3ENC-01/feats_vox2t3", "Experiment/T3ENC-01/feats_vox2lid", "Experiment/VOX2-01/feats_vox2"]
OPTIONAL_DIRS = {"Experiment/SCALE-VB/feats_vb": ["SCALE-VB", "VBSEL-01", "T3RECIPE-01"], "kaggle/output/feats_e6144": ["ENBASE-01"]}
with zipfile.ZipFile(io.BytesIO(base64.b64decode(CODE_B64))) as z:
    for f in z.namelist():
        b = z.read(f); assert hashlib.md5(b).hexdigest() == CODE_MD5[f], f
        (LAB / f).parent.mkdir(parents=True, exist_ok=True); (LAB / f).write_bytes(b)


def link(target, path):
    if path.exists():
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        os.symlink(target, path, target_is_directory=True)
    except OSError:                                                   # Windows without symlink rights (local smoke): junction
        subprocess.run(["cmd", "/c", "mklink", "/J", str(path), str(target)], check=True, capture_output=True)


for d in DATA_DIRS + list(OPTIONAL_DIRS):
    src = [r / d for r in ROOTS if (r / d).is_dir()]
    if not src:
        assert d not in DATA_DIRS and EXP not in OPTIONAL_DIRS.get(d, []), f"no attached lab bundle has {d} (needed by {EXP})"
        continue
    link(src[0], LAB / d)
for d, src in EXTRA_LINKS.items():                                   # features extracted above (FLAG_52 / 53)
    link(Path(src), LAB / d)
SAFE_SRC = E2E_DIR if (E2E_DIR / "safe04").is_dir() else next(d for d, m in [(m.parent, None) for m in IN.rglob("e2e_manifest.json")]
                                                              if (d / "safe04").is_dir())
for f in sorted((SAFE_SRC / "safe04").glob("*")):                     # reference tables of the gate (streams, rel, v11's T3, ...)
    shutil.copy(f, SP / "out/safe04" / f.name)
assert (LAB / "Experiment" / EXP / "run.py").exists(), f"{EXP} is not embedded in this notebook"
for f, h in CODE_MD5.items():
    print(f"  {h}  {f}")
ENVV = dict(os.environ, SAFE04_SCRATCH=str(SP), FLAG_LOCAL="1", FLAG_IN=str(eda[0].parent.parent), FLAG_OUT=str(SP / "out"),
            VOX_LID=str(LAB / "Experiment/T3ENC-01/feats_vox2lid/lid_summary.json"), E2E_DIR=str(E2E_DIR), E2E_SPLIT=SPLIT,
            E2E_OUT=str(OUTD), PYTHONUNBUFFERED="1", PYTHONIOENCODING="utf-8")
log(f"lab ready at {LAB}: {len(CODE_MD5)} code files; data: {[d for d in DATA_DIRS + list(OPTIONAL_DIRS) if (LAB / d).exists()]}; "
    f"{len(list((SP / 'out/safe04').glob('*')))} SAFE-04 reference tables")

## 2. Resume (optional)

In [ ]:
# resume: a previous version's e2e_out/ (same experiment) -> finished jobs and their files are copied back and skipped
PROG = OUTD / "progress.json"
prog = json.loads(PROG.read_text()) if PROG.exists() else {}
if not prog.get("done"):
    prevs = [] if LOCAL else [(q, json.loads(q.read_text())) for q in IN.rglob("e2e_out/progress.json")]
    prevs = [(q, p) for q, p in prevs if p.get("exp") == EXP and p.get("done")]
    # an output made by DIFFERENT code of this experiment (e.g. FLAG_51 before / after the SUB21a amendment of 11/10) is not
    # resumed: its finished folds would be mixed with folds of the new recipe
    changed = lambda p: [f for f in EXP_CODE if p.get("code_md5", {}).get(f) != CODE_MD5.get(f)]
    if os.environ.get("E2E_ALLOW_CODE_CHANGE") != "1":
        for q, p in prevs:
            if changed(p):
                log(f"!! NOT resuming from {q.parent}: made by different {EXP} code {changed(p)}; its folds would mix two "
                    "versions. This run starts fresh (E2E_ALLOW_CODE_CHANGE=1 overrides).")
        prevs = [(q, p) for q, p in prevs if not changed(p)]
    prevs.sort(key=lambda t: t[1].get("split") != SPLIT)           # the same split first
    for prev, p in prevs[:1]:
        same = p.get("split") == SPLIT
        for f in prev.parent.glob("*"):
            # another split (dev run -> Evaluation run): only the fold tables and the trained final models / heads move over;
            # the final job then reloads them and scores the new test files
            keep = same or f.name.endswith((".parquet", ".pt", "_selection.json"))
            if f.is_file() and f.name != "progress.json" and keep:
                shutil.copy(f, OUTD / f.name)
        for f in prev.parent.glob(f"{TAG}_fold*.parquet"):
            shutil.copy(f, SP / "out/safe04" / f.name)
        p = dict(p, split=SPLIT, done=[j for j in p["done"] if same or j.startswith("fold")])
        prog = p; log("resume from", prev.parent, f"(split {'same' if same else 'other'})", "jobs done", p["done"])
prog = prog if prog.get("exp") == EXP else {"exp": EXP, "split": SPLIT, "done": [], "seconds": {}, "files": {}}
prog["code_md5"] = CODE_MD5
PROG.write_text(json.dumps(prog, indent=1))

## 3. Jobs: SAFE-04 folds + final head, one per GPU at a time

In [ ]:
# JOBS: every SAFE-04 fold + the final head (started second, so it overlaps the folds); at most one subprocess per GPU;
# a job starts only if the budget allows it (elapsed < E2E_BUDGET_H and elapsed + 1.2 x the longest finished job < 11.5 h)
JOBLIST = [f"fold{s}" for s in FOLDS[:1]] + ([] if FINAL_AFTER_REPORT else ["final"]) + [f"fold{s}" for s in FOLDS[1:]]
todo = [j for j in JOBLIST if j not in prog["done"]]
slots = list(range(max(NGPU, 1)))
threads_per = str(max(1, (os.cpu_count() or 4) // len(slots)))
lock = threading.Lock()


def pump(job):
    for line in job["p"].stdout:
        job["tail"].append(line); del job["tail"][:-60]
        if "Warning" not in line and "warnings.warn" not in line:
            with lock:
                print(f"[{job['name']}] {line}", end="", flush=True)


def finish(job):
    name, t0 = job["name"], job["t0"]
    if job["p"].returncode != 0:
        print("".join(job["tail"])); raise RuntimeError(f"{EXP} job {name} failed (exit {job['p'].returncode})")
    files = sorted(SP.glob(f"out/safe04/{TAG}_{name}.parquet")) if name.startswith("fold") else []
    for f in files:
        shutil.copy(f, OUTD / f.name); prog["files"][f.name] = hashlib.md5(f.read_bytes()).hexdigest()
    prog["done"].append(name); prog["seconds"][name] = round(time.time() - t0)
    PROG.write_text(json.dumps(prog, indent=1))
    log(f"job {name} done [{time.time() - t0:.0f}s]")


def budget_ok():
    h = (time.time() - T_START) / 3600
    longest = max([v for v in prog["seconds"].values()] or [0]) / 3600
    return h < BUDGET_H and h + 1.2 * longest < 11.5


running = []
while todo or running:
    busy = {j["gpu"] for j in running}
    for g in [g for g in slots if g not in busy]:
        if not todo:
            break
        if not budget_ok():
            log(f"time budget: not starting {todo}"); todo = []; break
        name = todo.pop(0)
        env = dict(ENVV, SAFE04_THREADS=threads_per, CUDA_VISIBLE_DEVICES=str(g) if NGPU else "")
        args = [sys.executable, "-u", str(LAB / "lab_entry.py"), EXP] + (["final"] if name == "final" else [])
        if name.startswith("fold"):
            env["FOLDS"] = name[4:]
        p = subprocess.Popen(args, cwd=str(WORK), env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                             encoding="utf-8", errors="replace")
        job = dict(name=name, p=p, t0=time.time(), gpu=g, tail=[])
        job["th"] = threading.Thread(target=pump, args=(job,), daemon=True); job["th"].start()
        running.append(job)
        log(f"job {name} started on {'GPU ' + str(g) if NGPU else 'CPU'}")
    for job in list(running):
        if job["p"].poll() is not None:
            job["th"].join(); running.remove(job); finish(job)
    time.sleep(2)
missing = [j for j in JOBLIST if j not in prog["done"]]
if missing:
    log(f"!! time budget reached: jobs left {missing}. Save Version, attach this output as input, run again (finished jobs are skipped).")
else:
    log("all jobs done:", prog["seconds"])

## 4. Registered gate (in the notebook)

In [ ]:
# registered gate, computed here (Experiment/E2E/e2e_lib.gate through the experiment's report()); needs every fold
GATE_TXT = OUTD / f"{TAG}_report.txt"
if all(f"fold{s}" in prog["done"] for s in (1, 2, 3, 4, 5)):
    r = subprocess.run([sys.executable, "-u", str(LAB / "lab_entry.py"), EXP, "report"], cwd=str(WORK), env=ENVV,
                       capture_output=True, text=True, encoding="utf-8", errors="replace")
    print("\n".join(l for l in r.stdout.splitlines() if "Warning" not in l)); assert r.returncode == 0, r.stderr[-3000:]
    shutil.copy(SP / "out/safe04" / f"{TAG}_report.txt", GATE_TXT)
else:
    print(f"gate not computed: folds done {[j for j in prog['done'] if j.startswith('fold')]} (all 5 are needed)")

## 5. Final fuser + submission.zip

In [ ]:
# final fuser (CA-FUSE refit on every SAFE-04 row with the new stream) -> submission.zip; English files = v11
if "final" in prog["done"] and all(f"fold{s}" in prog["done"] for s in (1, 2, 3, 4, 5)):
    r = subprocess.run([sys.executable, "-u", str(LAB / "lab_entry.py"), EXP, "zip"], cwd=str(WORK), env=ENVV,
                       capture_output=True, text=True, encoding="utf-8", errors="replace")
    print("\n".join(l for l in r.stdout.splitlines() if "Warning" not in l)); assert r.returncode == 0, r.stderr[-3000:]
    ZP = OUTD / f"submission_{TAG}_{SPLIT}.zip"
    shutil.copy(ZP, WORK / "submission.zip")
    log("submission:", ZP, "sha256", hashlib.sha256(ZP.read_bytes()).hexdigest()[:16])
else:
    print("zip not built: it needs the final head and the 5 folds (the fuser is refitted on the SAFE-04 rows with the new stream)")

## 6. Summary

In [ ]:
prog.update(gpu=GPU, torch=torch.__version__, lab_data=MAN["created"], e2e=E2E_MAN["created"], finished=time.strftime("%Y-%m-%d %H:%M"))
PROG.write_text(json.dumps(prog, indent=1))
print(json.dumps({k: v for k, v in prog.items() if k != "code_md5"}, indent=1))
if (OUTD / f"{TAG}_report.txt").exists():
    print("\n".join(l for l in (OUTD / f"{TAG}_report.txt").read_text(encoding="utf-8").splitlines() if "DECISION" in l))
print("NEXT: download /kaggle/working/e2e_out (report, fold parquets, final models, fuser, zip). Submit submission.zip to CodaBench "
      "ONLY if the DECISION line says PASSES, and read it with the rule registered in the experiment README.")